In [1]:
pip install requests beautifulsoup4 pandas matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests

# Fetch the main Remote OK page
url = "https://remoteok.com/remote-jobs"
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    print("Success! Downloaded the page.")
else:
    print(f"Error: {response.status_code}")

# Print first 500 characters of the HTML
print(response.text[:500])

Success! Downloaded the page.
<!doctype html><html lang="en" class="   pageType-frontpage  remoteok    minimize-header   catch-emails-enabled">	<head>
			


					<link rel="stylesheet" href="/global.css?1761481057">
					<script>
													var userIsAdmin=false;
											</script>
					<meta charset="UTF-8">
					<title>Remote Jobs in Programming, Design, Sales and more #OpenSalaries</title>
					<meta name="description" content="Looking for a remote job? Remote OK® is the #1 Remote Job Platform and has 1,129,438+ remot


In [3]:
import requests

url = "https://remoteok.com/remote-jobs"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

response = requests.get(url, headers=headers, timeout=30)

print("Status Code:", response.status_code)
print(response.text[:500])


Status Code: 200
<!doctype html><html lang="en" class="   pageType-frontpage  remoteok    minimize-header   catch-emails-enabled">	<head>
			


					<link rel="stylesheet" href="/global.css?1761481057">
					<script>
													var userIsAdmin=false;
											</script>
					<meta charset="UTF-8">
					<title>Remote Jobs in Programming, Design, Sales and more #OpenSalaries</title>
					<meta name="description" content="Looking for a remote job? Remote OK® is the #1 Remote Job Platform and has 1,129,456+ remot


In [4]:
import requests

url = "https://remoteok.com/api"
headers = {"User-Agent": "Mozilla/5.0"}

data = requests.get(url, headers=headers).json()

job_titles = []

# first item is metadata → skip it
for job in data[1:]:
    title = job.get("position")
    if title:
        job_titles.append(title)

# Print first 10
for i, title in enumerate(job_titles[:10], 1):
    print(f"{i}. {title}")

1. Art Director
2. Lead Omics Workflow Engineer
3. ERP Systems Manager
4. Copy of Senior AI ML Engineer Applied Machine Learning
5. Senior Software Engineer Backend Engineering
6. Account Manager
7. Senior Software Engineer Frontend Engineering
8. AI Developer 1551
9. Software Engineer Agent Infrastructure
10. Software Engineer


In [5]:
import requests

url = "https://remoteok.com/api"
headers = {"User-Agent": "Mozilla/5.0"}

data = requests.get(url, headers=headers).json()

jobs = []

# Skip first metadata item
for job in data[1:]:
    job_data = {}

    # Extract title
    job_data['title'] = job.get('position', 'N/A')

    # Extract company
    job_data['company'] = job.get('company', 'N/A')

    # Extract skills (tags)
    tags = job.get('tags', [])
    job_data['skills'] = ', '.join(tags) if isinstance(tags, list) else 'N/A'

    # Extract location
    job_data['location'] = job.get('location', 'N/A')

    # Extract URL
    job_data['url'] = job.get('url', 'N/A')

    jobs.append(job_data)

# Print first job as test
if jobs:
    for key, value in jobs[0].items():
        print(f"{key}: {value}")

title: Art Director
company: Digital Media Management
skills: director, design, training, software, growth, video, cloud, strategy, exec, management, lead, marketing, health, illustrator, digital nomad
location: United States
url: https://remoteOK.com/remote-jobs/remote-art-director-digital-media-management-1129446


In [7]:
import requests
import csv
import time
from datetime import datetime

# =========================
# Configuration
# =========================
API_URL = "https://remoteok.com/api"
HEADERS = {"User-Agent": "Mozilla/5.0"}
PAGE_SIZE = 100
TOTAL_PAGES = 5
OUTPUT_FILE = "remoteok_jobs_day4.csv"

# =========================
# Fetch data from API
# =========================
try:
    response = requests.get(API_URL, headers=HEADERS, timeout=30)
    response.raise_for_status()
    data = response.json()
except requests.exceptions.RequestException as e:
    print("Network error:", e)
    exit()
except ValueError:
    print("Error: Invalid JSON response")
    exit()

# First item is metadata → skip it
jobs_data = data[1:]

all_jobs = []

# =========================
# Pagination simulation
# =========================
for page in range(1, TOTAL_PAGES + 1):
    print(f"Scraping page {page}...")

    start = (page - 1) * PAGE_SIZE
    end = page * PAGE_SIZE
    page_jobs = jobs_data[start:end]

    for job in page_jobs:

        # -------- Skills (tags) --------
        tags = job.get("tags", [])
        skills = ", ".join(tags) if isinstance(tags, list) else "N/A"

        # -------- Job Type (derived from tags) --------
        job_type = "N/A"
        if isinstance(tags, list):
            for t in tags:
                if t.lower() in ["full-time", "part-time", "contract", "internship", "freelance"]:
                    job_type = t
                    break

        # -------- Date Posted (REAL posting date) --------
        epoch = job.get("epoch")
        date_posted = (
            datetime.fromtimestamp(epoch).strftime("%Y-%m-%d")
            if epoch else "N/A"
        )

        # -------- Job record --------
        job_record = {
            "title": job.get("position", "N/A"),
            "company": job.get("company", "N/A"),
            "location": job.get("location", "N/A"),
            "skills": skills,
            "job_type": job_type,
            "date_refreshed": date_posted,
            "url": job.get("url", "N/A")
        }

        all_jobs.append(job_record)

    time.sleep(1)

print(f"\nTotal jobs scraped: {len(all_jobs)}")

# =========================
# Export to CSV
# =========================
if all_jobs:
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=[
               "title",
                "company",
                "location",
                "skills",
                "job_type",
                "date_refreshed",
                "url"
            ]
        )
        writer.writeheader()
        writer.writerows(all_jobs)

    print(f"Data successfully saved to '{OUTPUT_FILE}'")
else:
    print("No data to save.")


Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...

Total jobs scraped: 98
Data successfully saved to 'remoteok_jobs_day4.csv'
